# Payment Memory를 사용하는 Research Agent

## 개요

이전 튜토리얼의 agent는 stateless였으므로 모든 session이 처음부터 시작되었습니다. 이 튜토리얼에서는 **AgentCore Memory**를 추가하여 agent가 여러 session에 걸쳐 정보를 축적하도록 합니다.

- 이미 비용을 지불하고 조사한 topic을 기억하여 동일한 data에 다시 결제하지 않음
- 사용자 preference 학습(budget 허용 범위, 관심 topic)
- 유용한 endpoint와 비용이 높은 endpoint 추적

### Payments + Memory 작동 방식

```
Session 1 (new user)                    Session 2 (returning user)
  │                                       │
  │ "Research renewable energy outlook"     │ "Research renewable energy AND AI market trends"
  │                                       │
  ├─► Pay $0.05 — renewable energy      ├─► Recall per topic:
  ├─► Return summary                       │     • renewable energy → already in memory ✓
  │                                       │     • AI market trends → not in memory ✗
  │                                       ├─► Skip payment for renewable energy (free)
  │                                       ├─► Pay $0.05 only for AI market trends
  │                                       ├─► Return both summaries + savings report
  │                                       │
  └─► Memory extracts:                  └─► Result: paid $0.05 instead of $0.10 — memory saved $0.05.
      • renewable energy researched ($0.05)
```

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)의 무료 USDC와 함께 Base Sepolia 또는 Solana Devnet을 사용합니다. Testnet USDC에는 실제 가치가 없습니다.


### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                        |
|:--------------------|:----------------------------------------------------------------|
| Tutorial type       | Conversational                                                  |
| Agent type          | Single                                                          |
| Agentic Framework   | Strands Agents                                                  |
| LLM model           | Anthropic Claude Sonnet                                         |
| Tutorial 구성 요소  | AgentCore payments, AgentCore Memory, AgentCorePaymentsPlugin   |
| 예제 난이도         | 중급                                                            |
| 사용 SDK            | bedrock-agentcore SDK, Strands Agents SDK                       |

## 사전 요구 사항

* Tutorial 00 완료(`.env` 존재)
* https://faucet.circle.com/의 testnet USDC를 Wallet에 입금

이 튜토리얼은 Tutorial 00에서 구성한 모든 wallet provider(Coinbase CDP 또는 Stripe/Privy)에서 작동합니다. 선택한 provider와 관계없이 agent 코드는 동일합니다.

AWS credentials에는 Tutorial 00에서 생성한 IAM 권한(`setup_payment_roles()`)이 필요합니다. Tutorial 00을 성공적으로 완료했다면 필요한 권한이 이미 있습니다.

In [ ]:
!pip install -r requirements.txt --quiet

## 1단계 — Config 불러오기

In [ ]:
import sys, os, json, time, uuid

sys.path.append("..")

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from utils import load_tutorial_env, print_summary, client_token

config = load_tutorial_env()
PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

if config.get("multi_provider"):
    PROVIDER = list(config["instruments"].keys())[0]
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
    CONNECTOR_ID = config["instruments"][PROVIDER]["connector_id"]
else:
    INSTRUMENT_ID = config["instrument_id"]
    CONNECTOR_ID = config.get("connector_id")
    PROVIDER = config.get("provider_type", "unknown")

MODEL_ID = os.environ.get("MODEL_ID", "us.anthropic.claude-sonnet-4-6")

print_summary(
    "Config",
    manager_arn=PAYMENT_MANAGER_ARN,
    provider=PROVIDER,
    instrument_id=INSTRUMENT_ID,
)

## 2단계 — Instrument 검증 및 Session 생성

Tutorial 00에서 Payment Manager, Connector, Instrument를 생성했습니다. 이들과 상호 작용할 SDK client를 생성하고 instrument가 ACTIVE인지 검증한 다음 이 task를 위한 새 session을 생성합니다.

In [ ]:
from bedrock_agentcore.payments import PaymentManager

# Tutorial 00의 기존 Payment Manager ARN을 래핑하는 SDK client
manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

# Instrument가 ACTIVE인지 검증
instr = manager.get_payment_instrument(user_id=USER_ID, payment_instrument_id=INSTRUMENT_ID)
instr_status = instr.get("status", "UNKNOWN")
assert instr_status == "ACTIVE", f"Instrument is {instr_status} — fund and delegate in Tutorial 00/03 first"

# 이 task를 위한 새 session 생성
sess_resp = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.20", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
SESSION_ID = sess_resp["paymentSessionId"]
print(f"✅ Instrument {INSTRUMENT_ID} is {instr_status}")
print(f"✅ Session: {SESSION_ID} (budget: $0.20)")

### 두 계층의 Budget 제어

Session budget($0.20)은 AgentCore payments가 API level에서 적용하는 **hard limit**입니다. LLM의 결정과 관계없이 agent는 이를 초과할 수 없습니다.

Memory는 그 위에 **soft optimization** 계층을 추가합니다. Agent는 과거 session data를 사용하여 budget 내에서 더 효율적으로 지출을 결정합니다. Memory가 없으면 중복 호출에 $0.20를 낭비할 수 있지만, Memory를 사용하면 이미 아는 내용을 건너뛰어 budget을 더 효율적으로 활용합니다.

| 계층 | 제어 항목 | 적용 주체 |
|-------|----------|-------------|
| **Session budget**($0.20, 60분 후 만료) | 초과할 수 없는 hard ceiling | AgentCore payments service |
| **Memory intelligence** | 중복 호출을 건너뛰는 soft optimization | Agent logic(system prompt + recall tool) |

Budget 적용은 구조적(IAM + API)이며 Memory는 여기에 intelligence를 추가합니다.

## 3단계 — Memory 생성

대화에서 fact를 추출하는 semantic strategy와 함께 AgentCore Memory를 사용합니다. Memory는 다음 내용을 저장합니다.
- 사용자가 조사한 topic
- Agent가 호출한 endpoint 및 비용
- 대화 중에 표현된 사용자 preference

In [ ]:
memory_ctl = boto3.client("bedrock-agentcore-control", region_name=REGION)
memory_data = boto3.client("bedrock-agentcore", region_name=REGION)

MEMORY_NAME = f"research_memory_{uuid.uuid4().hex[:8]}"

memory_resp = memory_ctl.create_memory(
    name=MEMORY_NAME,
    description="Research agent memory - tracks topics, costs, and preferences",
    eventExpiryDuration=30,
    memoryStrategies=[
        {
            "semanticMemoryStrategy": {
                "name": "ResearchFacts",
                "namespaceTemplates": [f"/actor/{USER_ID}/facts/"],
            }
        }
    ],
)
MEMORY_ID = memory_resp["memory"]["id"]
print(f"✅ Memory created: {MEMORY_ID}")
print(f"   Strategy: ResearchFacts (semantic extraction)")
print(f"   Namespace: /actor/{USER_ID}/facts/")

# Memory가 ACTIVE가 될 때까지 대기. CreateMemory는 즉시
# status=CREATING을 반환하지만 이후 record operation(batch_create, retrieve)에는 ACTIVE가 필요함
print("\n   Waiting for memory to become ACTIVE (usually 30-90s)...", flush=True)
elapsed = 0
while True:
    status = memory_ctl.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if status == "ACTIVE":
        print(f"   ✅ Memory is ACTIVE (after {elapsed}s)", flush=True)
        break
    if status == "FAILED":
        reason = memory_ctl.get_memory(memoryId=MEMORY_ID)["memory"].get("failureReason", "unknown")
        raise RuntimeError(f"Memory creation failed: {reason}")
    print(f"   status={status}, elapsed={elapsed}s, polling again in 10s...", flush=True)
    time.sleep(10)
    elapsed += 10

## 4단계 — Memory 초기 데이터 적재(재방문 사용자 시뮬레이션)

Agent와의 research history가 이미 있는 재방문 사용자를 시뮬레이션하도록 Memory를 미리 채웁니다. 이전 session에서 각각 $0.05(총 $0.10)의 비용이 든 두 research topic을 불러옵니다.

- **Seattle weather** — Query 1 설정(full memory hit)
- **Renewable energy market outlook** — Query 3 설정(partial hit: 이 topic은 Memory에서 가져오고 AI market trends는 새로 조회)

Query 2에서 불러올 내용이 있도록 `user_profile` 및 `tool_preference` record도 적재합니다.

In [ ]:
from datetime import datetime, timedelta, timezone

# Cached research가 항상 최신으로 보이도록 어제 날짜 사용. 그렇지 않으면
# agent가 Memory hit를 너무 오래된 것으로 판단하여 새 data에 다시 결제함
yesterday = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

hydration_records = [
    {
        "content": {
            "text": json.dumps(
                {
                    "type": "user_profile",
                    "interests": ["weather data", "renewable energy", "market research"],
                    "budget_preference": "moderate - prefers endpoints under $0.10 per call",
                    "style": "concise summaries with key data points",
                    "last_session_total_spent": "$0.10",
                }
            )
        },
        "namespace": f"/actor/{USER_ID}/facts/",
    },
    {
        "content": {
            "text": json.dumps(
                {
                    "type": "past_research",
                    "date": yesterday,
                    "topic": "weather data for Seattle",
                    "cost": "$0.05",
                    "endpoint_used": "weather-api ($0.05, accurate 7-day forecast)",
                    "result_summary": "Seattle: 58F, partly cloudy, rain expected Thursday",
                }
            )
        },
        "namespace": f"/actor/{USER_ID}/facts/",
    },
    {
        "content": {
            "text": json.dumps(
                {
                    "type": "past_research",
                    "date": yesterday,
                    "topic": "renewable energy market outlook",
                    "cost": "$0.05",
                    "endpoint_used": "energy-insights-api ($0.05, concise sector summary)",
                    "result_summary": "Global renewable capacity additions on track to exceed 560 GW in 2026; solar leading growth; grid storage and offshore wind project pipelines expanding into late 2026",
                }
            )
        },
        "namespace": f"/actor/{USER_ID}/facts/",
    },
    {
        "content": {
            "text": json.dumps(
                {
                    "type": "tool_preference",
                    "preferred": [
                        "weather-api - $0.05, fast and accurate",
                        "energy-insights-api - $0.05, good sector summaries",
                    ],
                    "avoid": ["premium-analytics - $0.50 per call, too expensive for this user"],
                }
            )
        },
        "namespace": f"/actor/{USER_ID}/facts/",
    },
]

ts = time.time()
records_to_create = []
for idx, rec in enumerate(hydration_records):
    records_to_create.append(
        {
            "requestIdentifier": f"hydrate_{idx:03d}",
            "content": rec["content"],
            "namespaces": [rec["namespace"]],
            "timestamp": ts + idx,
        }
    )

resp = memory_data.batch_create_memory_records(
    memoryId=MEMORY_ID,
    records=records_to_create,
)
print(f"✅ Hydrated {len(resp.get('successfulRecords', []))} memory records")
print(f"   Namespace: /actor/{USER_ID}/facts/")
print(f"   Past research dated {yesterday}: Seattle weather ($0.05), renewable energy outlook ($0.05)")
print(f"   Last session total: $0.10")
print(f"\n   Waiting 25s for indexing...", flush=True)
time.sleep(25)
print("   ✅ Ready for semantic search", flush=True)

## 5단계 — Agent 구축

Agent에는 세 가지 기능이 있습니다.
1. **PaymentsPlugin** — `http_request`가 유료 endpoint에 도달하면 x402 자동 결제
2. **Memory tool** — agent가 결제를 결정하기 전에 사용자 history 조회
3. **http_request** — 유료 endpoint 호출(plugin이 402 flow 처리)

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from strands_tools import http_request
from bedrock_agentcore.payments.integrations.strands import (
    AgentCorePaymentsPlugin,
    AgentCorePaymentsPluginConfig,
)

# 결제 플러그인
payment_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=INSTRUMENT_ID,
        payment_session_id=SESSION_ID,
        region=REGION,
        network_preferences_config=["eip155:84532", "base-sepolia"]
        if os.environ.get("NETWORK", "ETHEREUM") == "ETHEREUM"
        else ["solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1"],
    )
)


# Memory query tool - RetrieveMemoryRecords 사용(semantic search)
@tool
def recall_user_context(query: str) -> str:
    """Search the user's memory for relevant context before making paid calls.

    Use this to check:
    - Has the user asked about this topic before?
    - What did past sessions cost?
    - Which endpoints does the user prefer or avoid?

    Args:
        query: Natural language search (e.g., 'weather data', 'budget preference')

    Returns:
        JSON with matching memory records.
    """
    results = memory_data.retrieve_memory_records(
        memoryId=MEMORY_ID,
        namespace=f"/actor/{USER_ID}/facts/",
        searchCriteria={
            "searchQuery": query,
            "topK": 5,
        },
    )
    # RetrieveMemoryRecords는 'memoryRecordSummaries' 아래에 match 반환
    records = results.get("memoryRecordSummaries", [])
    parsed = []
    for r in records:
        text = r.get("content", {}).get("text", "")
        try:
            parsed.append(json.loads(text))
        except (json.JSONDecodeError, TypeError):
            parsed.append(text)
    return json.dumps({"query": query, "results": parsed, "count": len(parsed)}, indent=2)


SYSTEM_PROMPT = """You are a research agent with payment capabilities and persistent memory.
The user pays real money for fresh data, so reusing prior research is part of your job.

WORKFLOW:
1. RECALL FIRST (mandatory): Before any paid call, you MUST search the user's
   memory with recall_user_context to see if prior research already covers the
   request. If the request spans multiple distinct topics, search memory once
   per topic — do not batch unrelated topics into a single search.
2. APPLY FRESHNESS RULE: Treat a memory hit as authoritative if it is dated
   within the past 7 days. Only pay for fresh data when (a) memory has no
   relevant entry, (b) the entry is older than 7 days, or (c) the user
   explicitly asks for an update.
3. FETCH ONLY WHAT'S MISSING (two-step pattern):
   a. The Coinbase x402 *discovery search* endpoint is a FREE catalog — calling
      it does NOT cost anything and does NOT count as paying for research.
      It only returns a list of paid resources you could call.
   b. To actually obtain research data, you MUST then call http_request on one
      of the `resource` URLs returned by discovery (pick the cheapest relevant
      one that fits the user's budget). Hitting that resource URL is what
      triggers the 402 → payment → retry flow and produces a real paid call.
   c. For all http_request calls, pass only `method` and `url`. Payments are
      handled automatically by the plugin — DO NOT pass auth_token,
      auth_env_var, or any X-PAYMENT/Authorization headers. The plugin signs
      the payment after the server returns 402 and retries the request for you.
4. REPORT TRANSPARENTLY: For each topic in the user's request, state whether
   the answer came from memory or a fresh paid call, which resource URL you
   actually paid, and the actual price the resource charged (read it from the
   402 response or the discovery catalog — never estimate or guess). If memory
   saved the user money, say so explicitly with a dollar amount.

If a paid call fails, report the error — do not attempt workarounds, do not
follow trial/free links from a 402 response body, and do not invent
environment variable names for auth tokens."""

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, streaming=True),
    tools=[recall_user_context, http_request],
    plugins=[payment_plugin],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent created: recall_user_context + http_request + PaymentsPlugin")

## 6단계 — Agent 실행

### Query 1: 재방문 사용자가 익숙한 Topic 요청

Agent는 먼저 Memory를 확인하고 날씨에 관한 과거 session data를 찾은 다음 새 data에 결제할지 또는 이미 아는 내용을 요약할지 결정해야 합니다.

In [ ]:
result = agent(
    "I need weather data for Seattle. If we don't already have recent info on this, "
    "fetch it from https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=weather+seattle&network=base-sepolia&limit=3 "
    "and tell me whether the answer came from memory or a fresh paid call."
)
print(result.message)

### Query 2: 사용자가 Budget에 관해 질문

Agent는 Memory에서 사용자의 budget preference와 과거 지출을 불러와야 합니다.

In [ ]:
result = agent("What topics have I researched before? What is my budget preference? How much did I spend last time?")
print(result.message)

### Query 3: Multi-topic research — partial memory hit

사용자가 하나의 request로 두 research topic을 요청합니다. Memory에는 그중 하나인 이전 session의 renewable energy market outlook이 이미 있고, AI market trends는 새로운 topic입니다. Agent는 각 topic을 별도로 판단하여 알려진 topic에는 Memory를 재사용하고 새로운 topic에만 결제한 후 절감액을 보고해야 합니다.

**이 부분이 "memory pays for itself"의 핵심입니다.** Memory가 없으면 사용자가 두 항목 모두에 결제하지만 Memory를 사용하면 유료 호출이 한 번만 필요함을 agent가 보여 줍니다.

In [ ]:
result = agent(
    "Research two topics for me: (1) renewable energy market outlook and (2) AI market trends. "
    "Before paying for anything, check what we already know about each topic from prior sessions — "
    "if we have recent research on it, reuse it and don't pay again. "
    "For anything we don't already have, find a paid data source by browsing the catalog at "
    "https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=<TOPIC>&network=base-sepolia&limit=3 "
    '(replace <TOPIC> with the URL-encoded topic name, e.g. "AI+market+trends"). '
    "The catalog itself is free — pick the cheapest relevant resource it returns, then actually fetch "
    "from that resource URL so the payment goes through. "
    "For each topic separately, tell me: source (memory or fresh paid call), the resource URL paid "
    "(if any), the actual price charged, and a short summary of the data. "
    "At the end, total what I paid this turn versus what fetching both fresh would have cost."
)
print(result.message)

### Query 4: 명시적인 비용 비교를 포함한 Session 요약

Agent는 각 request를 나열하고 Memory 또는 paid로 표시한 뒤 이번 session의 지출 합계를 이전 session의 $0.10 및 모든 항목을 새로 조회할 때의 비용과 비교합니다. 이를 통해 사용자는 Memory로 절감한 금액을 정확히 확인할 수 있습니다.

In [ ]:
result = agent(
    "Recap this whole session for me. List each request I made, whether it was answered "
    "from memory or by paying for fresh data, and the cost of each. "
    "Then compare total session spend to my last session and to what fresh research on "
    "everything would have cost — be specific with dollar amounts so I can see exactly what memory saved me."
)
print(result.message)

## 7단계 — Session 지출 확인

In [ ]:
session_info = manager.get_payment_session(
    user_id=USER_ID,
    payment_session_id=SESSION_ID,
)
sess = session_info
print_summary(
    "Session Spend",
    session_id=SESSION_ID,
    available=sess.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
    budget_limit=sess.get("limits", {}).get("maxSpendAmount", "N/A"),
)

## 8단계 — Budget 적용 확인

Memory는 지출을 최적화하지만 **session budget은 hard limit**입니다. 이를 검증하기 위해 유료 x402 resource보다 작은 $0.0001의 session을 생성하고 agent가 실제로 유료 resource 조회를 시도하도록 합니다. LLM의 결정과 관계없이 AgentCore payments가 API level에서 결제를 거부합니다.

> Discovery search 자체는 무료이므로 항상 성공합니다. Budget 적용을 발생시키려면 agent가 discovery에서 반환한 *resource* URL을 호출해야 하며, 이 지점에서 402 → payment flow가 발생합니다.

In [ ]:
# 모든 유료 x402 resource보다 작은 budget으로 session 생성
# 확인된 최저가 resource는 $0.001이므로 $0.0001 budget은 반드시 거부됨
tiny_resp = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.0001", "currency": "USD"}},
    expiry_time_in_minutes=15,
)
tiny_session_id = tiny_resp["paymentSessionId"]
print(f"Tiny session: {tiny_session_id} (budget: $0.0001)")

# 동일한 plugin 구조를 재사용하고 tiny session을 가리키도록 설정
tiny_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=INSTRUMENT_ID,
        payment_session_id=tiny_session_id,
        region=REGION,
        network_preferences_config=["eip155:84532", "base-sepolia"],
    )
)

# Budget test agent가 실제로 유료 resource를 시도하도록 동일한 2단계 pattern을 제공
# 무료 discovery catalog만 호출하면 검증할 수 없음
TINY_SYSTEM_PROMPT = """You are a research agent. To fetch paid data on x402:
1. Call the discovery search URL (this is FREE — it just returns a catalog).
2. Pick a resource URL from the catalog and call THAT URL with http_request.
   Step 2 is what triggers the 402 → payment flow. The plugin handles payment
   automatically — pass only `method` and `url` to http_request.
If a paid call is rejected, report the exact error message verbatim. Do not
attempt workarounds or invent retry strategies."""

budget_test_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, streaming=True),
    tools=[http_request],
    plugins=[tiny_plugin],
    system_prompt=TINY_SYSTEM_PROMPT,
)

print("\nAttempting a paid resource fetch with $0.0001 budget...")
result = budget_test_agent(
    "Find a paid weather data resource on x402 by browsing the catalog at "
    "https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=weather&network=base-sepolia&limit=1 "
    "and then actually call the resource URL it returns to fetch the weather data. "
    "Report exactly what happened — including any payment errors verbatim."
)
print(result.message)
print("\n✅ Budget enforcement: the $0.0001 session cannot cover any Bazaar resource call.")
print("   This is structural — enforced by AgentCore payments at the API level, not by agent logic.")

## Payment Trace 보기

모든 payment에서 trace가 생성됩니다. Amazon CloudWatch GenAI Observability Dashboard에서 service가 생성한 telemetry인 payment 성공률, session 지출, transaction latency를 살펴보세요.

In [ ]:
print(f"🔍 View your agent traces: Amazon CloudWatch → GenAI Observability Dashboard")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")

## 실행 결과

Payments와 Memory를 결합한 research agent를 구축했습니다.

1. **Memory recall** — agent가 지출 전에 사용자 history 확인
2. **효율적인 지출** — Memory에 답이 있으면 중복 호출 생략
3. **Payments** — 새 data 호출에 대해 plugin이 x402 자동 처리
4. **투명성** — agent가 Memory에서 불러온 항목과 결제한 항목 보고

### Memory가 Payments에 추가하는 기능

| Memory 미사용 | Memory 사용 |
|---------------|-------------|
| Agent가 session마다 동일한 data에 다시 결제 | Agent가 Memory를 먼저 확인하고 새 data에만 결제 |
| Session 간 비용 정보 없음 | Agent가 현재와 과거 session 비용 비교 |
| 모든 session이 초기 상태로 시작 | Agent가 사용자 preference와 tool quality 기억 |

### 배포된 Agent의 Role 분리

이 Notebook은 AWS credentials로 로컬에서 실행됩니다. 배포 시 runtime process는 ProcessPaymentRole로 실행되며, plugin은 app backend에서 설정한 budget 내에서 agent를 대신해 `ProcessPayment`를 호출합니다. Runtime은 session 생성, limit 수정 또는 wallet provision을 할 수 없습니다. Agent(LLM)는 `ProcessPayment`를 직접 호출하지 않습니다. Memory tool과 payment plugin은 배포 시에도 동일하게 작동합니다. 전체 role 분리 구현은 Tutorial 02를 참조하세요.

Role 분리를 로컬에서 테스트하려면 assumed-role session을 SDK client에 전달합니다.

```python
from utils import assume_role
import boto3

# 앱 백엔드(ManagementRole)가 세션 생성
manager = PaymentManager(payment_manager_arn=ARN, region_name=REGION)
session = manager.create_payment_session(user_id=USER_ID, ...)

# Agent는 ProcessPaymentRole로 실행되며 ProcessPayment만 수행 가능
agent_session = assume_role(boto3.Session(), PROCESS_PAYMENT_ROLE_ARN, 'agent')
agent_manager = PaymentManager(
    payment_manager_arn=ARN, boto3_session=agent_session
)
# Plugin에 agent_manager 전달 - 세션을 생성하거나 예산을 수정할 수 없음
```

## 리소스 정리

이 튜토리얼에서 생성한 Memory resource는 삭제해야 합니다. Session은 자동으로 만료됩니다. Payment resource(Manager, Connector, Instrument)는 Tutorial 00에서 생성했으므로 여기서 삭제하지 마세요.

In [ ]:
# Memory resource 삭제
try:
    memory_ctl.delete_memory(memoryId=MEMORY_ID)
    print(f"✅ Deleted memory: {MEMORY_ID}")
except Exception as e:
    print(f"⚠️  {e}")

# 축하합니다!

Data에 결제하고 시간이 지날수록 더 많은 정보를 활용하는 research agent를 구축했습니다. 다음 과정: **Tutorial 07** — Multi-Agent Payment Orchestrator